In [7]:
pdf_path = "../resources/GA_61IW_615000240_2S_2024-25.pdf"

In [2]:
from langchain.document_loaders import PyPDFLoader
from langchain.chat_models import ChatOllama
from langchain.chains import create_extraction_chain

def extract_evaluation_criteria_with_llm(pdf_path):
    # Cargar PDF
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    
    # Combinar texto
    full_text = "\n".join([page.page_content for page in pages])
    
    # Esquema para extracción
    schema = {
        "properties": {
            "criteria_section": {
                "type": "string",
                "description": "El contenido completo de la sección 'Criterios de evaluación'"
            }
        },
        "required": ["criteria_section"]
    }
    
    # Configurar LLM y cadena de extracción
    llm = ChatOllama(
            model = "deepseek-r1:8b",
            temperature = 0
        )
    chain = create_extraction_chain(schema, llm)
    
    # Ejecutar extracción
    prompt = f"""
    Extrae el texto completo de la sección llamada "Criterios de evaluación" 
    o la sección numerada que contiene "Criterios de evaluación" en el siguiente documento.
    Incluye todas las subsecciones y contenido hasta que comience la siguiente sección principal.
    
    Texto del documento:
    {full_text}  # Limitado para evitar contextos muy grandes
    """
    
    result = chain.run(prompt)
    
    if result and len(result) > 0 and "criteria_section" in result[0]:
        return result[0]["criteria_section"]
    else:
        return "No se pudo extraer la sección"

In [5]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain.document_loaders import PyPDFLoader
from langchain.chat_models import ChatOllama

# 1. Define un esquema para la sección que quieres extraer
class SeccionGuiaDocente(BaseModel):
    """Sección específica extraída de una guía docente."""
    titulo_completo: str = Field(
        description="El título completo de la sección (ej. '7.2. Criterios de evaluación')"
    )
    contenido: str = Field(
        description="El contenido completo de la sección, incluyendo todo el texto hasta la siguiente sección"
    )

# 2. Crea un prompt específico para guiar la extracción
prompt = ChatPromptTemplate.from_messages([
    ("system", """
     Eres un experto en extraer secciones específicas de guías docentes universitarias.
     
     Tu tarea es identificar y extraer con precisión la sección "Criterios de evaluación", 
     que puede estar numerada de diferentes formas (como 7.2, 5.3, etc.) en diferentes documentos.
     
     Debes extraer tanto el título completo como todo el contenido de la sección, 
     hasta (pero sin incluir) el inicio de la siguiente sección principal.
     
     Sé extremadamente preciso y extrae EXACTAMENTE el texto como aparece en el documento.
     """),
    ("human", "{text}")
])

# 3. Función para procesar el PDF y extraer la sección
def extract_evaluation_criteria(pdf_path):
    # Cargar el documento
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    full_text = "\n".join([page.page_content for page in pages])
    
    # Configurar el modelo (puedes elegir diferentes modelos según necesites)
    llm = ChatOllama(
            model = "deepseek-r1:8b",
            temperature = 0
        )  # O puedes usar Anthropic, Ollama, etc.
    
    # Configurar el modelo para salida estructurada
    structured_llm = llm.with_structured_output(schema=SeccionGuiaDocente)
    
    # Invocar el modelo con el prompt
    result = structured_llm.invoke(prompt.invoke({"text": full_text}))
    
    # Devolver solo el contenido, o toda la estructura si lo prefieres
    return result.contenido  # O return result para obtener tanto título como contenido



In [14]:
from langchain.document_loaders import PyPDFLoader
from langchain_community.llms import Ollama
import re

def extract_evaluation_criteria(pdf_path):
    """
    Extrae la sección 'Criterios de evaluación' de una guía docente usando
    una combinación de regex y LLM.
    """
    # Cargar el PDF
    try:
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        full_text = "\n".join([page.page_content for page in pages])
    except Exception as e:
        return f"Error al cargar el PDF: {str(e)}"
    
    # Segundo intento: usar LLM para extracción
    llm = Ollama(model="deepseek-r1:8b", temperature=0)  # Usa el modelo que prefieras
    
    prompt = f"""
    Extrae el texto completo de la sección llamada "Criterios de evaluación" 
    o la sección numerada que contiene "Criterios de evaluación" en el siguiente documento.
    Incluye todas las subsecciones y contenido hasta que comience la siguiente sección principal.
    
    Texto del documento:
    {full_text} 
    """
    
    try:
        response = llm.invoke(prompt)
        # Limpiar posible texto adicional
        cleaned_response = response.strip()
        
        # Verificar si la respuesta contiene "Criterios de evaluación"
        
        return cleaned_response
    except Exception as e:
        return f"Error al procesar con LLM: {str(e)}"

In [15]:
# Ejemplo de uso
resultado = extract_evaluation_criteria(pdf_path)
print(resultado)

Multiple definitions in dictionary at byte 0x5d0fc for key /PageMode


<think>
Okay, so I'm trying to understand this document about the "Fundamentals of Software Engineering" course. Let me go through it step by step.

First, I see that it's a guide for students in the ETSISI (Escuela Técnica Superior de Ingenieros Industriales y Sistemas Informáticos) at the Universidad Politécnica de Madrid. The document is structured with several sections: objectives, syllabus, resources, and additional information.

The objectives of the course are to provide students with a solid understanding of software engineering concepts, practices, and techniques. It also aims to develop problem-solving skills, encourage teamwork, and prepare them for real-world software development challenges. That makes sense because software engineering is a broad field that covers many aspects beyond just coding.

Looking at the syllabus, it's divided into 14 modules. Each module seems to cover specific topics like software process models, requirements analysis, design, testing, etc. The f